# Campos Gerais Water: Published Contrasts and Weighting Sensitivity (v2)

Project lead: Lissandra Kruse Fuganti.

Source: de Paula et al. (2026), https://doi.org/10.1007/s11368-026-04517-0, Supplementary Table S1, CC BY 4.0. Attributed transcription converted to CSV; these measurements were not collected by this project author. Code developed with AI assistance; human review remains necessary.

**24 selected runoff-event records, 20 dates, 96 rounded percentage contrasts. Not raw time series, not 24 independent sites, and not all eligible rainfall events. No machine learning, causal inference, waterlogging prediction or regional validation. Unweighted medians are not pooled reductions. 100% is interpreted only as reported.**

This self-contained notebook requires only the Python standard library: no network, GPU, credentials or installations. Run all cells to generate JSON, CSV, a report and a SHA-256 manifest. Version 2 adds date-weighting sensitivity, leave-one-date-out influence diagnostics, and explicitly hypothetical denominator scenarios; it does not add field observations.

Repository: https://github.com/lissandrakruse/campos-gerais-agua


In [ ]:
from pathlib import Path
import sys
ROOT = Path("/kaggle/working/campos-gerais-agua") if Path("/kaggle/working").exists() else Path("campos-gerais-agua-notebook")
ROOT.mkdir(parents=True, exist_ok=True)
p=ROOT/'scripts/extract_springer_s1.py'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_bytes(b'#!/usr/bin/env python3\n"""Extract published derived contrasts, NOT raw runoff, from de Paula et al. S1.\nInput: official DOCX; output: sourced CSV and descriptive audit (no inference).\n"""\nimport argparse,csv,datetime,hashlib,json,re,statistics,zipfile\nfrom pathlib import Path\nimport xml.etree.ElementTree as ET\nDOI=\'10.1007/s11368-026-04517-0\'\nURL=\'https://media.springernature.com/original/springer-static/esm/art%3A10.1007%2Fs11368-026-04517-0/MediaObjects/11368_2026_4517_MOESM1_ESM.docx\'\nMETRICS=[\'ssc_med_difference_pct\',\'ssc_max_difference_pct\',\'trv_difference_pct\',\'tsy_difference_pct\']\nNS={\'w\':\'http://schemas.openxmlformats.org/wordprocessingml/2006/main\'}\ndef extract(path):\n root=ET.fromstring(zipfile.ZipFile(path).read(\'word/document.xml\'))\n tables=root.findall(\'.//w:tbl\',NS)\n assert len(tables)==1, \'Review changed source: expected one table\'\n rows=[]\n for tr in tables[0].findall(\'w:tr\',NS)[3:]:\n  cells=[\'\'.join(t.itertext()).strip() for t in tr.findall(\'w:tc\',NS)]\n  assert len(cells)==5\n  label=cells[0];datepart,_,suffix=label.partition(\'_\')\n  date=datetime.datetime.strptime(datepart,\'%b %d, %Y\').date().isoformat()\n  row={\'source_row\':len(rows)+1,\'event_label_original\':label,\'event_date\':date,\'event_suffix\':suffix}\n  for name,value in zip(METRICS,cells[1:]): row[name]=float(value.replace(\',\',\'.\'))\n  row[\'source_doi\']=DOI;row[\'source_table\']=\'Supplementary Table S1\';rows.append(row)\n assert len(rows)==24\n return rows\ndef describe(rows):\n dates={r[\'event_date\'] for r in rows}\n out={\'n_event_rows\':len(rows),\'n_distinct_dates\':len(dates),\'duplicate_full_event_labels\':len(rows)-len({r[\'event_label_original\'] for r in rows}), \'multiple_events_same_date\':{d:sum(r[\'event_date\']==d for r in rows) for d in sorted(dates) if sum(r[\'event_date\']==d for r in rows)>1},\'year_counts\':{y:sum(r[\'event_date\'][:4]==y for r in rows) for y in sorted({r[\'event_date\'][:4] for r in rows})},\'metrics\':{}}\n for m in METRICS:\n  a=[r[m] for r in rows];out[\'metrics\'][m]={\'n\':len(a),\'min_pct\':min(a),\'max_pct\':max(a),\'median_pct_unweighted\':statistics.median(a),\'n_negative\':sum(x<0 for x in a),\'n_zero\':sum(x==0 for x in a),\'n_positive\':sum(x>0 for x in a),\'n_equal_100_reported\':sum(x==100 for x in a)}\n out[\'interpretation\']=\'Published rounded relative contrasts only. No absolute runoff volumes, rainfall predictors or sampling denominators. Unweighted descriptions are not pooled reductions, confidence intervals or independent treatment effects. 100% is as reported, not independently verified absence.\'\n return out\ndef main():\n p=argparse.ArgumentParser();p.add_argument(\'docx\',type=Path);p.add_argument(\'--out\',type=Path,default=Path(\'.\'));a=p.parse_args();a.out.mkdir(parents=True,exist_ok=True)\n rows=extract(a.docx)\n with (a.out/\'de_paula_2026_s1_derived_event_contrasts.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n  w=csv.DictWriter(f,fieldnames=list(rows[0]),lineterminator="\\n");w.writeheader();w.writerows(rows)\n summary=describe(rows);summary[\'source_sha256\']=hashlib.sha256(a.docx.read_bytes()).hexdigest();summary[\'source_url\']=URL\n (a.out/\'de_paula_2026_s1_audit.json\').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+\'\\n\')\n print(json.dumps(summary,indent=2))\nif __name__==\'__main__\':main()\n')
p=ROOT/'scripts/analyze_sensitivity.py'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_bytes(b'#!/usr/bin/env python3\n"""Descriptive weighting audit; no causal/predictive inference or observed denominators."""\nimport argparse,csv,hashlib,json,statistics\nfrom collections import defaultdict\nfrom pathlib import Path\nMETRICS=[\'ssc_med_difference_pct\',\'ssc_max_difference_pct\',\'trv_difference_pct\',\'tsy_difference_pct\']\ndef weighted_mean(v,w):return sum(x*y for x,y in zip(v,w))/sum(w)\ndef box_extrema(v,k):\n    """Exact fractional-linear extrema for independently bounded positive weights [1,k].\n    An optimum has boundary weights; ordered thresholds suffice by exchange argument.\n    Endpoints allowing all 1/all k include the constant-weight case.\n    """\n    if k<1:raise ValueError(\'k must be >= 1\')\n    s=sorted(v);n=len(s)\n    lo=min(weighted_mean(s,[k if i<j else 1 for i in range(n)]) for j in range(n+1))\n    hi=max(weighted_mean(s,[1 if i<j else k for i in range(n)]) for j in range(n+1))\n    return lo,hi\n\ndef analyze(path,out):\n    raw=path.read_bytes();rows=list(csv.DictReader(raw.decode().splitlines()))\n    groups=defaultdict(list)\n    for r in rows:groups[r[\'event_date\']].append(r)\n    summaries=[];lodo=[];bounds=[];date_rows=[]\n    for m in METRICS:\n        v=[float(r[m]) for r in rows]\n        dm={d:statistics.mean(float(r[m]) for r in rr) for d,rr in groups.items()}\n        for d,x in dm.items():date_rows.append({\'metric\':m,\'date\':d,\'record_count\':len(groups[d]),\'within_date_mean_pct\':x})\n        p=sum(x for x in v if x>0);a=-sum(x for x in v if x<0)\n        for d in sorted(groups):\n            kept=[float(r[m]) for r in rows if r[\'event_date\']!=d]\n            lodo.append({\'metric\':m,\'excluded_date\':d,\'remaining_records\':len(kept),\'row_mean_pct\':statistics.mean(kept),\'row_median_pct\':statistics.median(kept),\'date_balanced_mean_pct\':statistics.mean(x for dd,x in dm.items() if dd!=d)})\n        lm=[x for x in lodo if x[\'metric\']==m]\n        summaries.append({\'metric\':m,\'n_records\':len(v),\'n_dates\':len(dm),\'n_positive\':sum(x>0 for x in v),\'n_negative\':sum(x<0 for x in v),\'row_mean_pct\':statistics.mean(v),\'row_median_pct\':statistics.median(v),\'date_balanced_mean_pct\':statistics.mean(dm.values()),\'date_balanced_median_pct\':statistics.median(dm.values()),\'date_minus_row_mean_pp\':statistics.mean(dm.values())-statistics.mean(v),\'lodo_row_mean_min_pct\':min(x[\'row_mean_pct\'] for x in lm),\'lodo_row_mean_max_pct\':max(x[\'row_mean_pct\'] for x in lm),\'unknown_weight_closure_min_pct\':min(v) if m.startswith((\'trv\',\'tsy\')) else None,\'unknown_weight_closure_max_pct\':max(v) if m.startswith((\'trv\',\'tsy\')) else None,\'denominator_ratio_sign_threshold\':p/a if a and m==\'tsy_difference_pct\' else None})\n        for k in ([1,2,5,6,10,100] if m.startswith((\'trv\',\'tsy\')) else []):\n            lo,hi=box_extrema(v,k);bounds.append({\'metric\':m,\'assumed_max_to_min_denominator_ratio\':k,\'lower_weighted_contrast_pct\':lo,\'upper_weighted_contrast_pct\':hi,\'interpretation\':\'hypothetical independently bounded denominators; not estimated physical values\'})\n    witnesses=[]\n    for scenario in [\'equal\',\'negative_tsy_weight_6\']:\n        for r in rows:\n            d=float(r[\'tsy_difference_pct\']);a=6.0 if scenario==\'negative_tsy_weight_6\' and d<0 else 1.0\n            witnesses.append({\'scenario\':scenario,\'source_row\':r[\'source_row\'],\'event_date\':r[\'event_date\'],\'reported_tsy_contrast_pct\':d,\'constructed_nt_arbitrary_units\':a,\'constructed_terraced_arbitrary_units\':a*(1-d/100),\'status\':\'constructed compatibility witness, NOT observed sediment yields\'})\n    witness_summaries=[]\n    for scenario in [\'equal\',\'negative_tsy_weight_6\']:\n        rr=[r for r in witnesses if r[\'scenario\']==scenario];a=sum(r[\'constructed_nt_arbitrary_units\'] for r in rr);b=sum(r[\'constructed_terraced_arbitrary_units\'] for r in rr)\n        witness_summaries.append({\'scenario\':scenario,\'constructed_pooled_contrast_pct\':100*(a-b)/a,\'recovered_percentages_max_absolute_error\':max(abs(100*(r[\'constructed_nt_arbitrary_units\']-r[\'constructed_terraced_arbitrary_units\'])/r[\'constructed_nt_arbitrary_units\']-r[\'reported_tsy_contrast_pct\']) for r in rr)})\n    out.mkdir(parents=True,exist_ok=True)\n    for name,data in [(\'metric_sensitivity.csv\',summaries),(\'leave_one_date_out.csv\',lodo),(\'date_balanced_inputs.csv\',date_rows),(\'denominator_bound_scenarios.csv\',bounds),(\'constructed_tsy_witnesses.csv\',witnesses)]:\n        target = out / (\'constructed_scenarios\' if name in [\'denominator_bound_scenarios.csv\',\'constructed_tsy_witnesses.csv\'] else \'published_descriptions\') / name\n        target.parent.mkdir(parents=True,exist_ok=True)\n        with target.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            w=csv.DictWriter(f,fieldnames=list(data[0]),lineterminator="\\n");w.writeheader();w.writerows(data)\n    result={\'source_sha256\':hashlib.sha256(raw).hexdigest(),\'source_doi\':\'10.1007/s11368-026-04517-0\',\'records\':len(rows),\'dates\':len(groups),\'same_date_multiple_records\':{d:len(rr) for d,rr in groups.items() if len(rr)>1},\'metrics\':summaries,\'constructed_witness_summary\':witness_summaries,\'scope\':\'Published rounded contrasts, selected records at one paired site; no p-values, confidence intervals, generalization or observed absolute denominators.\'}\n    (out/\'sensitivity_results.json\').write_text(json.dumps(result,indent=2,allow_nan=False)+\'\\n\')\n    return result\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--input\',type=Path,required=True);p.add_argument(\'--out\',type=Path,required=True);a=p.parse_args();print(json.dumps(analyze(a.input,a.out),indent=2))\n')
p=ROOT/'scripts/run_pipeline.py'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_bytes(b'#!/usr/bin/env python3\n"""Deterministic descriptive pipeline for the published S1 contrasts only."""\nimport argparse, csv, datetime, hashlib, json, math\nfrom pathlib import Path\nfrom extract_springer_s1 import describe, METRICS, DOI, URL\nfrom analyze_sensitivity import analyze\nEXPECTED_CSV_SHA256 = "52ff69ca53da06c2bd2d441629b79efe64a5435973d9c1957f97096cfbf10215"\nDEFAULT_INPUT = Path(__file__).resolve().parents[1] / "data/published_derived/de_paula_2026_s1_derived_event_contrasts.csv"\nCAVEAT = "Published rounded percentage contrasts, not raw measurements. Unweighted medians are not pooled reductions. No predictive model, causal inference, or regional validation. These 24 selected runoff-event records cover 20 dates of one experiment, not 24 independent sites or all eligible storms. Reported 100% is not independently verified absence. Outlet runoff is not within-field waterlogging. Occurrence classification requires all eligible storms, including verified zero-runoff events."\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\ndef validate(path):\n    # Exact bytes pin all 96 numeric cells, event labels and attribution to audited version.\n    if sha(path) != EXPECTED_CSV_SHA256:\n        raise ValueError("CSV differs from the audited published table; re-audit required")\n    with Path(path).open(encoding="utf-8", newline="") as f:\n        rows = list(csv.DictReader(f))\n    if len(rows) != 24:\n        raise ValueError("Expected 24 source rows")\n    for i, row in enumerate(rows, 1):\n        if int(row["source_row"]) != i or row["source_doi"] != DOI:\n            raise ValueError("Invalid source provenance")\n        datetime.date.fromisoformat(row["event_date"])\n        row["source_row"] = i\n        for m in METRICS:\n            row[m] = float(row[m])\n            if not math.isfinite(row[m]):\n                raise ValueError("Nonfinite source value")\n    return rows\n\ndef run(input_path=DEFAULT_INPUT, out=Path("outputs/published")):\n    input_path, out = Path(input_path), Path(out)\n    rows = validate(input_path)\n    summary = describe(rows)\n    out.mkdir(parents=True, exist_ok=True)\n    (out / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True)+"\\n", encoding="utf-8")\n    fields = ["metric"] + list(summary["metrics"][METRICS[0]])\n    with (out / "metrics.csv").open("w", encoding="utf-8", newline="") as f:\n        writer=csv.DictWriter(f,fieldnames=fields); writer.writeheader()\n        for m in METRICS: writer.writerow({"metric":m, **summary["metrics"][m]})\n    lines=["# Descriptive reproduction of Supplementary Table S1", "", CAVEAT, "", f"Source: https://doi.org/{DOI} (Supplementary Table S1; CC BY 4.0).", "", "24 event rows, 20 dates, 96 percentage values checked against the audited version.", "", "| Metric | Minimum (%) | Maximum (%) | Unweighted median (%) | Negative values |", "|---|---:|---:|---:|---:|"]\n    for m in METRICS:\n        s=summary["metrics"][m]\n        lines.append(f"| {m} | {s[\'min_pct\']} | {s[\'max_pct\']} | {s[\'median_pct_unweighted\']} | {s[\'n_negative\']} |")\n    lines += ["", "No observed pooled reduction is calculated: absolute denominators are missing. Constructed denominator scenarios are explicitly hypothetical. No confidence intervals or machine-learning performance are calculated. Original extraction and limitations are documented in data/published_derived/PROVENANCE.md and docs/dados.md.", "", "Row counts do not measure spatial replication. Date inconsistencies between papers are not automatically reconciled. Future limited models may use audited event aggregates and predictors; minute-level hydrographs are not required for every model."]\n    (out / "report.md").write_text("\\n".join(lines)+"\\n",encoding="utf-8")\n    analyze(input_path, out / "sensitivity")\n    output_names=["summary.json","metrics.csv","report.md"] + sorted(str(p.relative_to(out)) for p in (out/"sensitivity").rglob("*") if p.is_file())\n    manifest={"pipeline_version":"2.0.0", "input_sha256":sha(input_path), "source_doi":DOI,"source_url":URL,"license_derived_table":"CC BY 4.0", "mode":"published_derived_and_separate_constructed_scenarios", "outputs":{n:sha(out/n) for n in output_names}}\n    # No wall-clock timestamps or absolute paths: same bytes in every output directory.\n    (out / "manifest.json").write_text(json.dumps(manifest,indent=2,sort_keys=True)+"\\n",encoding="utf-8")\n    return summary\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--input", type=Path, default=DEFAULT_INPUT)\n    p.add_argument("--out", type=Path, default=Path("outputs/published"))\n    a=p.parse_args(); s=run(a.input,a.out)\n    print(f"Validated {s[\'n_event_rows\']} event rows; descriptive outputs: {a.out}")\nif __name__ == "__main__": main()\n')
p=ROOT/'data/published_derived/de_paula_2026_s1_derived_event_contrasts.csv'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_bytes(b'source_row,event_label_original,event_date,event_suffix,ssc_med_difference_pct,ssc_max_difference_pct,trv_difference_pct,tsy_difference_pct,source_doi,source_table\n1,"Feb 17, 2023",2023-02-17,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n2,"Mar 2, 2023_T",2023-03-02,T,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n3,"Mar 3, 2023_N",2023-03-03,N,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n4,"Mar 3, 2023",2023-03-03,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n5,"Apr 16, 2023",2023-04-16,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n6,"Apr 18, 2023",2023-04-18,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n7,"Jul 27, 2023",2023-07-27,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n8,"Aug 11, 2023",2023-08-11,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n9,"Sep 2, 2023",2023-09-02,,88.04,87.8,57.14,99.82,10.1007/s11368-026-04517-0,Supplementary Table S1\n10,"Oct 8, 2023",2023-10-08,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n11,"Oct 12, 2023_T",2023-10-12,T,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n12,"Oct 12, 2023_N",2023-10-12,N,5.19,35.6,64.29,80.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n13,"Oct 26, 2023",2023-10-26,,100.0,100.0,96.55,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n14,"Oct 27, 2023",2023-10-27,,-18.24,-27.82,60.09,42.5,10.1007/s11368-026-04517-0,Supplementary Table S1\n15,"Oct 28, 2023_T",2023-10-28,T,67.51,98.5,85.24,95.65,10.1007/s11368-026-04517-0,Supplementary Table S1\n16,"Oct 28, 2023_N",2023-10-28,N,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n17,"Oct 29, 2023",2023-10-29,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n18,"Oct 30, 2023",2023-10-30,,46.38,-62.28,81.44,87.87,10.1007/s11368-026-04517-0,Supplementary Table S1\n19,"Dec 4, 2023",2023-12-04,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n20,"Nov 28, 2024",2024-11-28,,-118.93,-14.72,41.35,-9.66,10.1007/s11368-026-04517-0,Supplementary Table S1\n21,"Dec 7, 2024",2024-12-07,,13.68,-30.68,35.99,64.72,10.1007/s11368-026-04517-0,Supplementary Table S1\n22,"Dec 9, 2024_T1",2024-12-09,T1,41.62,55.94,25.89,-347.22,10.1007/s11368-026-04517-0,Supplementary Table S1\n23,"Dec 9, 2024_T2",2024-12-09,T2,-35.46,-8.63,44.17,27.22,10.1007/s11368-026-04517-0,Supplementary Table S1\n24,"Jan 31, 2025",2025-01-31,,-53.43,-172.21,30.74,-2.27,10.1007/s11368-026-04517-0,Supplementary Table S1\n')
p=ROOT/'docs/sensitivity_method.md'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_bytes(b"# Event-weighting and denominator-identifiability audit\n\n## Scope and provenance\n\nThis is a new secondary computation using the 96 reported two-decimal percentage contrasts in Supplementary Table S1 of de Paula et al. (2026), DOI https://doi.org/10.1007/s11368-026-04517-0. The source CSV SHA256 is recorded in the output. It is not a new field experiment, new predictive model, new mathematical theorem, or correction of the source authors' findings. The source study has additional measurements and aggregate results that are not encoded in Table S1. Our non-identifiability statement concerns reuse of **Table S1 alone**, not everything known by the authors or reported elsewhere.\n\n24 records have 20 calendar dates: four dates each contain two labeled records. Labels are retained. The two macroplots at one site are not independent treatment replicates. Grouping by date is a weighting sensitivity and minimum anti-leakage precaution, not proof that the remaining dates are independent storms. Consecutive-day storms and carry-over sediment can require broader grouping; metadata are insufficient to define that grouping here.\n\n## Reproducible computations\n\n1. Unweighted record means and medians for each of four variables.\n2. Date-balanced mean: first calculate an unweighted mean within date, then average those 20 means equally. This changes the descriptive estimand, not the physical water/sediment accounting.\n3. Leave-one-date-out: remove all rows sharing one date and recompute summaries. Report the range across 20 exclusions. These ranges are influence diagnostics, **not confidence intervals** or predictive validation.\n4. Counts of positive/negative contrasts and their conditional relationship to aggregate direction.\n5. For additive TRV and TSY only, independently bounded-denominator scenarios and a constructed non-identifiability witness. Do not interpret sums of event SSC maxima or means as cumulative sediment concentration, so these aggregate scenarios are deliberately not calculated for SSC.\n\n## Conditional aggregation identity\n\nFor selected, non-overlapping event records with the same units/normalization basis and strictly positive non-terraced quantities A_i, let B_i be the terraced quantity and d_i = 100(A_i-B_i)/A_i. Then the corresponding selected-event pooled contrast is D = 100(sum A_i - sum B_i)/sum A_i = sum(w_i d_i), where w_i = A_i/sum A_i. This is an elementary weighted-mean identity.\n\nIf all that is known is d_i, the scale and weights are unavailable. The closure of admissible D values is [min d_i, max d_i]; with all weights strictly positive and heterogeneous d, the endpoints are limits rather than attainable extrema. This is not an annual budget or a population interval. The reported percentages are rounded; all numerical bounds here use the published values as input. They are not exact bounds on unknown unrounded physical quantities.\n\nFor TRV all 24 reported contrasts are positive, ranging 25.89\xe2\x80\x93100%. Consequently any admissible positive weighting preserves the direction for these selected records, although it does not identify the magnitude. For TSY there are 21 positives and three negatives, ranging -347.22\xe2\x80\x93100%; therefore direction is not identified by these percentages alone.\n\n## Bounded-denominator scenarios\n\nAssume, for sensitivity analysis only, 1 <= A_i <= k after arbitrary common rescaling, without additional constraints across records. No observed or literature-based k is claimed. Fractional-linear extrema occur at bounds. Ordering d_i allows checking n+1 threshold allocations for minimum and maximum: give weight k to a prefix of the smallest contrasts (minimum) or a suffix of largest contrasts (maximum). Exhaustive corner tests on small vectors and interior checks independently verify implementation.\n\nFor TSY, the smallest denominator ratio permitting a nonpositive pooled contrast is k* = sum(d_i>0 d_i) / [-sum(d_i<0 d_i)] = 5.284087428650981. At k*, assigning k* to negative rows and 1 to positive rows gives zero numerator; below k*, no allocation within the independent box gives a negative numerator. This is an algebraic stress-test threshold, **not an estimate of the actual denominator ratio**.\n\nAt k=5, exact box bounds are 2.8342\xe2\x80\x9390.0058%; at k=6, -7.2285\xe2\x80\x9391.4998%. These are assumed-weight scenario ranges, not statistical uncertainty intervals. The specific simple witness below is compatible but not necessarily the minimum at k=6.\n\n## Constructed witnesses, NOT observed sediment yields\n\n- Assign every A_i=1 arbitrary unit: recover every reported percentage with B_i=A_i(1-d_i/100), pooled TSY contrast +64.1096%.\n- Assign A_i=6 to the three negative-TSY rows and A_i=1 to other rows: same 24 reported percentages, pooled TSY contrast -6.5928%.\n\nAll constructed B_i are nonnegative. The maximum recovered-percentage error is <6e-14 percentage points. These examples demonstrate why those percentages alone do not determine a pooled total. They do not assert that either denominator configuration occurred, that the configurations fit all information in the full paper, or that the source study's aggregate conclusion is wrong. No synthetic value is included in the observed-data CSV.\n\n## Observed descriptive results\n\n| Variable | Record mean (%) | Date-balanced mean (%) | Change (percentage points) | Leave-one-date-out record-mean range (%) |\n|---|---:|---:|---:|---:|\n| SSCmed |59.8483|59.8465|-0.0018|56.1982\xe2\x80\x9367.6213|\n| SSCmax |56.7292|53.5398|-3.1894|52.7955\xe2\x80\x9366.6830|\n| TRV |80.1204|80.6548|+0.5343|78.3132\xe2\x80\x9384.2195|\n| TSY |64.1096|70.5403|+6.4307|60.8468\xe2\x80\x9384.4832|\n\nMedians of the 24 records are 100% for all four metrics, illustrating that a median is not a pooled reduction. Date-balanced TRV median is98.275%; this statistic also is not a pooled budget.\n\n## Scientific value and novelty boundary\n\nThe credible contribution is a tested, source-linked **data-reuse audit** exposing which directions survive missing denominator weights, how grouping changes summaries, and which metadata must be recovered before a stronger hydrological model can be evaluated. This is more substantive than simply reprinting percentages, but it does not establish broad novelty or guarantee publication.\n\nTemporal aggregation and sediment weighting are established issues. Primary examples checked:\n- Lu, H., Moran, C. J., and Sivapalan, M. (2005), *A theoretical exploration of catchment-scale sediment delivery*, Water Resources Research, DOI https://doi.org/10.1029/2005WR004018. The paper explicitly treats different temporal averaging windows; two corrections are linked by the publisher. We do not reproduce or rely on its corrected equations.\n- Sk\xc3\xa5lev\xc3\xa5g, A., Korup, O., and Bronstert, A. (2024), *Inferring sediment-discharge event types in an Alpine catchment from sub-daily time series*, HESS28,4771ff, DOI https://doi.org/10.5194/hess-28-4771-2024. It calculates event sediment yield by integrating concentration\xc3\x97discharge, illustrating why individual concentration percentages cannot replace loads.\n\nThis was a focused prior-art check, not a systematic or exhaustive novelty search. An extension to other independent published sites/tables would be needed to assess transferability; the current study is a single-data-source case study.\n\n## Run\n\npython scripts/run_pipeline.py --out outputs/published\npython -m unittest discover -s tests -v\n\nFive tests pass, including exact comparison with exhaustive bound corners, interior-value enclosure, source-summary checks, witness recovery, sign threshold and byte-identical repeated outputs. Tests establish software properties, not field validity.\n")
sys.path.insert(0,str(ROOT/"scripts"))
from run_pipeline import run
summary=run(ROOT/"data/published_derived/de_paula_2026_s1_derived_event_contrasts.csv",ROOT/"outputs/published")
print((ROOT/"outputs/published/report.md").read_text())


In [ ]:
import json
manifest=json.loads((ROOT/"outputs/published/manifest.json").read_text())
print(json.dumps(manifest,indent=2))


## Limitations and next steps

The pipeline pins the audited CSV with a hash; it does not independently validate instruments or the original experiment. For a fresh extraction, obtain the official DOCX through the DOI supplementary-material link and run the repository extractor. Changed input requires a fresh audit. The CSV changes decimal commas to decimal points and adds ISO dates while preserving source labels and attribution.

The contrasts are reported as 100 × (NT − NT+T) / NT, comparing non-terraced and terraced no-till plots. Negative values are retained. Absolute runoff volumes and rainfall predictors cannot be reconstructed from percentages alone. Same-date events and both plots cannot be treated as independent validation samples. Date inconsistencies across papers require review before joining sources.

These are selected runoff records. Occurrence classification would need the full eligible storm set, including verified zero-runoff events. Audited event aggregates with predictors may support a limited future model; minute-level hydrographs are not mandatory for every model. Independent validation is still required. Post-event vegetation images cannot serve as pre-event predictors.

Synthetic examples belong to a separate schema-validation flow and are rejected here. No operational irrigation recommendations are made. The CC BY 4.0 attribution applies to the derived source table; it is not a blanket license for every project artifact.


## Secondary analysis: observed published contrasts

Date-balanced summaries average within each date first. Leave-one-date-out ranges are influence diagnostics, **not confidence intervals or predictive validation**. Grouping by date does not prove independence of storms. These records come from one paired site.

In [ ]:
sensitivity=json.loads((ROOT/"outputs/published/sensitivity/sensitivity_results.json").read_text())
for m in sensitivity["metrics"]:
    print(m["metric"], "record mean:", round(m["row_mean_pct"],4), "date-balanced mean:", round(m["date_balanced_mean_pct"],4), "percentage points difference:",round(m["date_minus_row_mean_pp"],4))


## Constructed denominator scenarios — NOT observed sediment yields

For additive quantities only, the selected-event pooled contrast is a denominator-weighted mean of event contrasts, conditional on consistent units and non-overlapping event records. Missing denominator weights in **Table S1 alone** prevent recovery of a unique pooled magnitude. This is not a correction of the source study; the full paper contains additional measurements and aggregate results.

TRV signs remain positive under positive weights for these selected records. TSY has mixed signs. The following **constructed compatibility witnesses in arbitrary units** recover the same published TSY percentages while giving different pooled contrasts. They are not physical estimates, do not necessarily fit all information in the full paper, and are stored separately from published descriptions. Independently bounded-weight scenarios are assumptions, not estimated uncertainty intervals. SSC is excluded from additive-total scenarios.

In [ ]:
for scenario in sensitivity["constructed_witness_summary"]:
    print("CONSTRUCTED ONLY:",scenario["scenario"],"pooled contrast (%):",round(scenario["constructed_pooled_contrast_pct"],4),"recovery error:",scenario["recovered_percentages_max_absolute_error"])
print("Full method and limitations:")
print((ROOT/"docs/sensitivity_method.md").read_text())
